# Блок 4 • Занятие 1 — выборка, статистики, устойчивость к выбросам (TODO)
**Дата:** 2026-03-02  
**Формат:** мини-теория → практика в коде → автопроверка (`assert`) → Run all  
**Цель занятия:** научиться считать базовые статистики по выборке и понимать разницу “параметр vs статистика”.

---

## Мини-теория (коротко, но по делу)

### 1) Что такое выборка
- **Генеральная совокупность** — “все возможные данные” (обычно недоступно целиком).
- **Выборка** — часть данных, которую мы реально собрали (из файла/БД/API).  
В дипломе вы почти всегда работаете именно с **выборкой**.

### 2) Параметр vs статистика
- **Параметр** — истинная характеристика совокупности (например, “реальное среднее по всем пользователям”).
- **Статистика** — оценка по выборке (например, среднее по вашим 500 записям).

### 3) Главные статистики
- `mean` — среднее (чувствительно к выбросам)
- `median` — медиана (устойчивее к выбросам)
- `variance/std` — разброс (вариация, стандартное отклонение)

### 4) Почему это нужно для диплома
Это ваш “модуль качества данных”:
- увидеть выбросы и шум
- выбрать правильные метрики
- объяснить пользователю/заказчику, почему вы доверяете данным

---

## Как работать
- Запускайте ячейки сверху вниз, или **Runtime → Run all**
- В конце должно быть: **✅ BLOCK04 LESSON01: all tests passed**

---

## Задача урока (перенос в VS Code)
В конце занятия перенесите функции в файл проекта:
- `src/math_stats.py`: `mean`, `median`, `variance_sample`, `std_sample`, `trimmed_mean`  


---
## Ячейка 1/10 — данные как список чисел, `len`, `print`

**Что изучаем здесь**
- `list[float]` — список чисел (выборка)
- `len(x)` — размер выборки `n`
- `print(...)` — вывести результат


In [1]:
sample = [4,2,6,3,4,2,1,5,3,1]

print("sample:", sample)
print("n =", len(sample))


sample: [4, 2, 6, 3, 4, 2, 1, 5, 3, 1]
n = 10


---
## Ячейка 2/10 — минимум/максимум: `min`, `max`, `sorted`

**Что изучаем здесь**
- `min(values)` и `max(values)` — крайние значения
- `sorted(values)` — сортировка (нужна для медианы)


In [2]:
print("min =", min(sample))
print("max =", max(sample))
print("sorted =", sorted(sample))


min = 1
max = 6
sorted = [1, 1, 2, 2, 3, 3, 4, 4, 5, 6]


---
## Ячейка 3/10 — среднее: `sum`, `len`, функция `mean`

**Термин**
- Среднее (mean) = сумма / количество


In [3]:
def mean(values: list[float]) -> float:
    """Среднее арифметическое. Требует непустой список."""
    if len(values) == 0:
        raise ValueError("mean: empty list")
    return sum(values) / len(values)

print("mean =", mean(sample))


mean = 3.1


---
## Ячейка 4/10 — медиана: `sorted`, индексы, чёт/нечёт

**Термин**
- Медиана — “середина” отсортированных данных.


In [4]:
def median(values: list[float]) -> float:
    """Медиана. Требует непустой список."""
    if len(values) == 0:
        raise ValueError("median: empty list")
    s = sorted(values)
    n = len(s)
    mid = n // 2
    if n % 2 == 0:
        return (s[mid - 1] + s[mid]) / 2
    else:
        return float(s[mid])

print("median =", median(sample))


median = 3.0


---
## Ячейка 5/10 — выборочная дисперсия: сумма квадратов отклонений

Формула: `sum((x-mean)^2) / (n-1)`


In [5]:
def variance_sample(values: list[float]) -> float:
    """Выборочная дисперсия (деление на n-1)."""
    n = len(values)
    if n < 2:
        raise ValueError("variance_sample: need at least 2 values")
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / (n - 1)
print("variance_sample =", variance_sample(sample))


variance_sample = 2.766666666666667


---
## Ячейка 6/10 — стандартное отклонение: корень из дисперсии

`std = variance ** 0.5`


In [6]:
def std_sample(values: list[float]) -> float:
    """Выборочное стандартное отклонение."""
    return variance_sample(values) ** 0.5

print("std_sample =", std_sample(sample))


std_sample = 1.66332999331662


---
## Ячейка 7/10 — выбросы: сравним mean и median “до/после”

Выброс добавляем в новый список, исходный не меняем.


In [7]:
def with_outlier(values: list[float], outlier: float) -> list[float]:
    """Вернуть новую выборку, добавив выброс (не меняем исходный список)."""
    return list(values) + [outlier]

sample_out = with_outlier(sample, 100)

print("mean(before) =", round(mean(sample), 3), "median(before) =", median(sample))
print("mean(after)  =", round(mean(sample_out), 3), "median(after)  =", median(sample_out))


mean(before) = 3.1 median(before) = 3.0
mean(after)  = 11.909 median(after)  = 3.0


---
## Ячейка 8/10 — trimmed mean: устойчивое среднее

Убираем `k` самых маленьких и `k` самых больших значений.


In [8]:
def trimmed_mean(values: list[float], k: int = 1) -> float:
    """Усечённое среднее: убрать k минимальных и k максимальных."""
    n = len(values)
    if n == 0:
        raise ValueError("trimmed_mean: empty list")
    if 2 * k >= n:
        raise ValueError("trimmed_mean: k too large")
    s = sorted(values)
    core = s[k:n - k]
    return mean(core)

print("trimmed_mean(before) =", round(trimmed_mean(sample, k=1), 3))
print("trimmed_mean(after)  =", round(trimmed_mean(sample_out, k=1), 3))


trimmed_mean(before) = 3.0
trimmed_mean(after)  = 3.333


---
## Ячейка 9/10 — функция `describe`: мини-отчёт в словаре (dict)

Это удобно для логирования, БД и отчётов в дипломе.


In [9]:
def describe(values: list[float]) -> dict:
    """Короткое описание выборки (как мини-отчёт)."""
    return {
        "n": len(values),
        "min": min(values) if values else None,
        "max": max(values) if values else None,
        "mean": mean(values) if values else None,
        "median": median(values) if values else None,
        "std": std_sample(values) if len(values) >= 2 else None,
    }

print("describe(sample) =", describe(sample))
print("describe(sample_out) =", describe(sample_out))


describe(sample) = {'n': 10, 'min': 1, 'max': 6, 'mean': 3.1, 'median': 3.0, 'std': 1.66332999331662}
describe(sample_out) = {'n': 11, 'min': 1, 'max': 100, 'mean': 11.909090909090908, 'median': 3.0, 'std': 29.259031239788325}


---
## Ячейка 10/10 — автопроверка (`assert`)

В solved-версии все тесты должны пройти.  
В TODO-версии тесты пройдут после того, как вы замените `# TODO` на рабочий код.


In [10]:
# =========================
# BLOCK 04 — LESSON 01 TESTS (НЕ МЕНЯТЬ)
# =========================

def approx(a: float, b: float, eps: float = 1e-9) -> bool:
    return abs(a - b) <= eps

def run_all_tests():
    s = [10, 11, 9, 10, 12, 9, 10, 11, 10, 9]

    assert len(s) == 10
    assert min(s) == 9
    assert max(s) == 12

    assert approx(mean(s), 10.1)
    assert median(s) == 10.0

    # mean=10.1, sum((x-m)^2)=8.9 => /9
    assert approx(round(variance_sample(s), 6), round(8.9/9, 6))
    assert approx(round(std_sample(s), 6), round((8.9/9) ** 0.5, 6))

    s2 = with_outlier(s, 100)
    assert len(s2) == 11
    assert approx(round(mean(s2), 6), round((sum(s)+100)/11, 6))
    assert median(s2) == 10.0  # медиана устойчива

    assert approx(trimmed_mean([1, 2, 3, 100], k=1), 2.5)
    assert trimmed_mean([1, 2, 3, 4, 100, 101], k=1) == mean([2, 3, 4, 100])

    d = describe(s)
    assert d["n"] == 10
    assert d["min"] == 9 and d["max"] == 12
    assert approx(d["mean"], 10.1)
    assert d["median"] == 10.0
    assert d["std"] is not None

    print("✅ BLOCK04 LESSON01: all tests passed")

run_all_tests()


✅ BLOCK04 LESSON01: all tests passed
